### 绘制图表

In [16]:
from __future__ import annotations

from pathlib import Path
from typing import List, Optional, Dict, Tuple
import re
import html

import numpy as np
import pandas as pd
import plotly.graph_objects as go


# =========================
# Config
# =========================
PER_IMAGE_DIR = Path("./_eval_exports_per_images") / "sahi_null_v3"
FO_BASE = "https://fiftyone.tianqiyao.men"
OUT_ROOT = PER_IMAGE_DIR / "_prod_kiss_reports"

LIMIT: Optional[int] = None
TOP_N = 20

FOCUS_COL = "focus"  # ✅ 你说列名就是 focus


# =========================
# Utils
# =========================
def find_per_image_csvs(root: Path) -> List[Path]:
    return sorted(root.rglob("image_level_*.csv"))


def safe_name(s: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9._-]+", "_", str(s))
    return s[:200]


def make_fo_url(dataset_name: str, sample_id: str, fo_base: str = FO_BASE) -> str:
    return f"{fo_base}/datasets/{dataset_name}?id={sample_id}"


def ensure_time_cols(df: pd.DataFrame) -> pd.DataFrame:
    if "capture_datetime" in df.columns:
        df["capture_datetime"] = pd.to_datetime(df["capture_datetime"], errors="coerce")

    if "capture_date" not in df.columns or df["capture_date"].isna().all():
        if "capture_datetime" in df.columns:
            df["capture_date"] = df["capture_datetime"].dt.date.astype(str)
        else:
            df["capture_date"] = "unknown_date"
    else:
        df["capture_date"] = df["capture_date"].astype(str)

    return df


def ensure_event_cols(df: pd.DataFrame) -> pd.DataFrame:
    for c in ["gt_count_img", "pred_count_img", "tp_img", "fp_img", "fn_img"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    has_gt = "gt_count_img" in df.columns
    has_pred = "pred_count_img" in df.columns
    pred_present = (df["pred_count_img"] > 0) if has_pred else pd.Series(False, index=df.index)

    if has_gt:
        gt_present = df["gt_count_img"] > 0
        df["hit_img"] = (gt_present & pred_present).astype(int)
        df["miss_img"] = (gt_present & ~pred_present).astype(int)
        df["false_alarm_img"] = (~gt_present & pred_present).astype(int)
        df["correct_reject_img"] = (~gt_present & ~pred_present).astype(int)
    else:
        df["hit_img"] = pred_present.astype(int)
        df["miss_img"] = 0
        df["false_alarm_img"] = 0
        df["correct_reject_img"] = 0

    return df


def add_fo_url(df: pd.DataFrame) -> pd.DataFrame:
    must = {"dataset_name", "sample_id"}
    if must.issubset(df.columns):
        df["sample_id"] = df["sample_id"].astype(str)
        df["url"] = df.apply(lambda r: make_fo_url(r["dataset_name"], r["sample_id"], FO_BASE), axis=1)
    else:
        df["url"] = ""
    return df


def infer_event_label(df: pd.DataFrame) -> pd.Series:
    has_gt = "gt_count_img" in df.columns
    if has_gt:
        cond_hit = df.get("hit_img", 0).astype(int) == 1
        cond_miss = df.get("miss_img", 0).astype(int) == 1
        cond_fa = df.get("false_alarm_img", 0).astype(int) == 1
        cond_cr = df.get("correct_reject_img", 0).astype(int) == 1

        out = pd.Series(["Unknown"] * len(df), index=df.index)
        out[cond_hit] = "Hit"
        out[cond_miss] = "Miss"
        out[cond_fa] = "False Alarm"
        out[cond_cr] = "Correct Reject"
        return out

    pred_present = pd.to_numeric(df.get("pred_count_img", 0), errors="coerce").fillna(0) > 0
    return pd.Series(["Hit" if b else "No-Hit" for b in pred_present], index=df.index)


def _x_for_day(df_day: pd.DataFrame):
    df_day = df_day.copy()
    if "capture_datetime" in df_day.columns:
        df_day["capture_datetime"] = pd.to_datetime(df_day["capture_datetime"], errors="coerce")
        if df_day["capture_datetime"].notna().any():
            return df_day["capture_datetime"], "capture_datetime", df_day
    df_day["_index"] = range(len(df_day))
    return df_day["_index"], "index", df_day


def _make_hover_cols(df: pd.DataFrame) -> List[str]:
    cols = []
    for c in [
        "focus",
        "filepath",
        "capture_datetime",
        "gt_count_img",
        "pred_count_img",
        "tp_img",
        "fp_img",
        "fn_img",
        "avg_confidence",
        "median_confidence",
        "confidence_threshold",
        "iou_threshold",
        "model_tag",
        "event",
        "err_obj",
        "tp_ratio",
    ]:
        if c in df.columns:
            cols.append(c)
    return cols


def _build_hovertemplate(y_label: str, hover_cols: List[str]) -> str:
    lines = [f"<b>{y_label}</b>: %{{y}}<br>"]
    for i, c in enumerate(hover_cols, start=1):
        lines.append(f"{c}: %{{customdata[{i}]}}<br>")
    lines.append("<b>Click</b> to open FiftyOne<br>")
    return "".join(lines)


def _get_focus_values(df: pd.DataFrame, focus_col: str) -> List[str]:
    if focus_col not in df.columns:
        return []
    vals = df[focus_col].astype(str).fillna("unknown").unique().tolist()
    vals = [v for v in vals if v.strip() != ""]
    return sorted(vals)


def _scope_prefix(scope: str) -> str:
    return f"{scope} | "


def _scope_for_focus_value(f: str) -> str:
    return f"focus={f}"


def _add_numeric_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in ["gt_count_img", "pred_count_img", "tp_img", "fp_img", "fn_img", "avg_confidence"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    return df


# =========================
# Top tables blocks (TOTAL + per-focus), each tagged with data-scope
# =========================
def make_top_tables_blocks_html(
    df_day: pd.DataFrame,
    top_n: int = 20,
    focus_col: str = "focus",
) -> str:
    d0 = df_day.copy()
    d0 = add_fo_url(d0)

    if focus_col not in d0.columns:
        d0[focus_col] = "unknown"
    d0[focus_col] = d0[focus_col].astype(str).fillna("unknown")

    for c in ["pred_count_img", "gt_count_img", "avg_confidence", "tp_img", "fp_img", "fn_img", "tp_ratio"]:
        if c in d0.columns:
            d0[c] = pd.to_numeric(d0[c], errors="coerce")

    d0["event"] = infer_event_label(d0)
    d0["err_obj"] = pd.to_numeric(d0.get("fp_img", 0), errors="coerce").fillna(0) + \
                    pd.to_numeric(d0.get("fn_img", 0), errors="coerce").fillna(0)

    def esc(x: object) -> str:
        return html.escape("" if pd.isna(x) else str(x))

    def short_path(p: str, maxlen: int = 70) -> str:
        p = "" if pd.isna(p) else str(p)
        if len(p) <= maxlen:
            return p
        return "…" + p[-(maxlen - 1):]

    headers = ["Event", "focus", "capture_datetime", "filepath", "gt", "pred", "tp", "fp", "fn", "fp+fn", "tp/gt", "avg_conf", "Link"]

    def build_table(df_sub: pd.DataFrame, scope: str, title: str) -> str:
        dd = df_sub.copy()
        dd["_err"] = pd.to_numeric(dd.get("err_obj", 0), errors="coerce").fillna(0)
        dd["_fn"] = pd.to_numeric(dd.get("fn_img", 0), errors="coerce").fillna(0)
        dd["_fp"] = pd.to_numeric(dd.get("fp_img", 0), errors="coerce").fillna(0)
        dd = dd.sort_values(["_err", "_fn", "_fp"], ascending=[False, False, False]).head(top_n)

        rows = []
        for _, r in dd.iterrows():
            url = r.get("url", "")
            link = f'<a href="{esc(url)}" target="_blank">Open</a>' if url else ""

            cap = r.get("capture_datetime", "")
            cap = "" if pd.isna(cap) else str(cap)

            tp_ratio = r.get("tp_ratio", "")
            tp_ratio = "" if pd.isna(tp_ratio) else f"{float(tp_ratio):.3f}"

            rows.append([
                esc(r.get("event", "")),
                esc(r.get(focus_col, "")),
                esc(cap),
                esc(short_path(r.get("filepath", ""), 70)),
                esc(r.get("gt_count_img", "")) if "gt_count_img" in dd.columns else "",
                esc(r.get("pred_count_img", "")) if "pred_count_img" in dd.columns else "",
                esc(r.get("tp_img", "")) if "tp_img" in dd.columns else "",
                esc(r.get("fp_img", "")) if "fp_img" in dd.columns else "",
                esc(r.get("fn_img", "")) if "fn_img" in dd.columns else "",
                esc(r.get("err_obj", "")),
                esc(tp_ratio),
                esc(r.get("avg_confidence", "")),
                link,
            ])

        buf = []
        buf.append(f"<div class='top_table_block' data-scope='{esc(scope)}' style='display:none;'>")
        buf.append(f"<h3 style='margin-top:18px;'>{esc(title)}</h3>")
        buf.append("<div style='overflow:auto; max-height:460px; border:1px solid #ddd;'>")
        buf.append("<table style='border-collapse:collapse; width:100%; font-family:Arial; font-size:12px;'>")
        buf.append("<thead><tr>")
        for h in headers:
            buf.append(
                f"<th style='position:sticky; top:0; background:#f7f7f7; border:1px solid #ddd; padding:6px; text-align:left;'>{esc(h)}</th>"
            )
        buf.append("</tr></thead><tbody>")
        for row in rows:
            buf.append("<tr>")
            for cell in row:
                buf.append(f"<td style='border:1px solid #ddd; padding:6px; vertical-align:top;'>{cell}</td>")
            buf.append("</tr>")
        buf.append("</tbody></table></div></div>")
        return "\n".join(buf)

    focuses = sorted(d0[focus_col].unique().tolist())

    blocks = []
    blocks.append(build_table(d0, scope="TOTAL", title=f"TOTAL (all focus) | Top {top_n} by (fp+fn)"))
    for f in focuses:
        scope = _scope_for_focus_value(f)
        sub = d0[d0[focus_col] == f]
        blocks.append(build_table(sub, scope=scope, title=f"{scope} | Top {top_n} by (fp+fn)"))

    return "\n".join(blocks)


def make_day_summary_html(df_day: pd.DataFrame) -> str:
    def fnum(x: object, nd: int = 3) -> str:
        try:
            if pd.isna(x):
                return "-"
            return f"{float(x):.{nd}f}"
        except Exception:
            return "-"

    n = len(df_day)
    gt_total = df_day["gt_count_img"].sum() if "gt_count_img" in df_day.columns else pd.NA
    pred_total = df_day["pred_count_img"].sum() if "pred_count_img" in df_day.columns else pd.NA
    tp_total = df_day["tp_img"].sum() if "tp_img" in df_day.columns else pd.NA
    fp_total = df_day["fp_img"].sum() if "fp_img" in df_day.columns else pd.NA
    fn_total = df_day["fn_img"].sum() if "fn_img" in df_day.columns else pd.NA

    block = []
    block.append("<div style='border:1px solid #ddd; background:#fafafa; padding:12px; margin:10px 0 14px 0; font-family:Arial;'>")
    block.append("<div style='font-size:14px; font-weight:700; margin-bottom:6px;'>Day summary</div>")
    block.append("<div style='font-size:12px; line-height:1.6;'>")
    block.append(f"Images: <b>{n}</b><br>")
    if "gt_count_img" in df_day.columns:
        block.append(f"GT total / avg: <b>{int(gt_total)}</b> / <b>{fnum(gt_total/max(n,1))}</b><br>")
    if "pred_count_img" in df_day.columns:
        block.append(f"Pred total / avg: <b>{int(pred_total)}</b> / <b>{fnum(pred_total/max(n,1))}</b><br>")
    if "tp_img" in df_day.columns:
        block.append(f"TP total / avg: <b>{int(tp_total)}</b> / <b>{fnum(tp_total/max(n,1))}</b><br>")
    if "fp_img" in df_day.columns:
        block.append(f"FP total / avg: <b>{int(fp_total)}</b> / <b>{fnum(fp_total/max(n,1))}</b><br>")
    if "fn_img" in df_day.columns:
        block.append(f"FN total / avg: <b>{int(fn_total)}</b> / <b>{fnum(fn_total/max(n,1))}</b><br>")
    block.append("</div></div>")
    return "\n".join(block)


# =========================
# Image-level day page: one global dropdown controls ALL plots + table
# =========================
def write_image_level_day_html(
    df_day: pd.DataFrame,
    out_html: Path,
    title: str,
    top_n: int = 20,
    focus_col: str = "focus",
) -> None:
    df_day = df_day.copy()
    df_day = add_fo_url(df_day)

    if focus_col not in df_day.columns:
        df_day[focus_col] = "unknown"
    df_day[focus_col] = df_day[focus_col].astype(str).fillna("unknown")

    x_all, x_title, df_day = _x_for_day(df_day)

    df_day = _add_numeric_cols(df_day)
    df_day["event"] = infer_event_label(df_day)
    df_day["err_obj"] = df_day.get("fp_img", 0).fillna(0) + df_day.get("fn_img", 0).fillna(0)

    if "gt_count_img" in df_day.columns and "tp_img" in df_day.columns:
        gt = pd.to_numeric(df_day["gt_count_img"], errors="coerce").fillna(0)
        tp = pd.to_numeric(df_day["tp_img"], errors="coerce").fillna(0)
        df_day["tp_ratio"] = np.where(gt > 0, tp / gt, np.nan)
    else:
        df_day["tp_ratio"] = pd.NA

    hover_cols = _make_hover_cols(df_day)

    focus_values = _get_focus_values(df_day, focus_col)
    scopes = ["TOTAL"] + [_scope_for_focus_value(f) for f in focus_values]

    def add_scope_traces(fig: go.Figure, scope: str, dsub: pd.DataFrame, xsub, which: str):
        custom = dsub[["url"] + hover_cols].values
        prefix = _scope_prefix(scope)

        if which == "fig0":
            if "gt_count_img" in dsub.columns:
                fig.add_trace(go.Scatter(
                    x=xsub, y=dsub["gt_count_img"], mode="lines+markers",
                    name=prefix + "GT count",
                    customdata=custom,
                    hovertemplate=_build_hovertemplate(prefix + "GT count", hover_cols),
                    visible=(scope == "TOTAL"),
                ))
            if "pred_count_img" in dsub.columns:
                fig.add_trace(go.Scatter(
                    x=xsub, y=dsub["pred_count_img"], mode="lines+markers",
                    name=prefix + "Pred count",
                    customdata=custom,
                    hovertemplate=_build_hovertemplate(prefix + "Pred count", hover_cols),
                    visible=(scope == "TOTAL"),
                ))

        elif which == "fig1":
            if "gt_count_img" in dsub.columns:
                fig.add_trace(go.Scatter(
                    x=xsub, y=dsub["gt_count_img"], mode="lines+markers",
                    name=prefix + "GT count",
                    customdata=custom,
                    hovertemplate=_build_hovertemplate(prefix + "GT count", hover_cols),
                    visible=(scope == "TOTAL"),
                ))
            if "tp_img" in dsub.columns:
                fig.add_trace(go.Scatter(
                    x=xsub, y=dsub["tp_img"], mode="lines+markers",
                    name=prefix + "TP",
                    customdata=custom,
                    hovertemplate=_build_hovertemplate(prefix + "TP", hover_cols),
                    visible=(scope == "TOTAL"),
                ))
            if "tp_ratio" in dsub.columns and pd.to_numeric(dsub["tp_ratio"], errors="coerce").notna().any():
                fig.add_trace(go.Scatter(
                    x=xsub, y=dsub["tp_ratio"], mode="lines+markers",
                    name=prefix + "TP/GT",
                    yaxis="y2",
                    customdata=custom,
                    hovertemplate=_build_hovertemplate(prefix + "TP/GT", hover_cols),
                    visible=(scope == "TOTAL"),
                ))

        elif which == "fig2":
            for col, nm in [("fp_img", "FP"), ("fn_img", "FN"), ("err_obj", "FP+FN")]:
                if col in dsub.columns:
                    fig.add_trace(go.Scatter(
                        x=xsub, y=dsub[col], mode="lines+markers",
                        name=prefix + nm,
                        customdata=custom,
                        hovertemplate=_build_hovertemplate(prefix + nm, hover_cols),
                        visible=(scope == "TOTAL"),
                    ))

    fig0 = go.Figure()
    fig1 = go.Figure()
    fig2 = go.Figure()

    add_scope_traces(fig0, "TOTAL", df_day, x_all, "fig0")
    add_scope_traces(fig1, "TOTAL", df_day, x_all, "fig1")
    add_scope_traces(fig2, "TOTAL", df_day, x_all, "fig2")

    for f in focus_values:
        scope = _scope_for_focus_value(f)
        sub = df_day[df_day[focus_col] == f].copy()
        xsub, _, sub = _x_for_day(sub)

        sub = _add_numeric_cols(sub)
        sub["event"] = infer_event_label(sub)
        sub["err_obj"] = sub.get("fp_img", 0).fillna(0) + sub.get("fn_img", 0).fillna(0)

        if "gt_count_img" in sub.columns and "tp_img" in sub.columns:
            gt = pd.to_numeric(sub["gt_count_img"], errors="coerce").fillna(0)
            tp = pd.to_numeric(sub["tp_img"], errors="coerce").fillna(0)
            sub["tp_ratio"] = np.where(gt > 0, tp / gt, np.nan)
        else:
            sub["tp_ratio"] = pd.NA

        add_scope_traces(fig0, scope, sub, xsub, "fig0")
        add_scope_traces(fig1, scope, sub, xsub, "fig1")
        add_scope_traces(fig2, scope, sub, xsub, "fig2")

    fig0.update_layout(
        title=f"{title} | GT vs Pred (global focus dropdown)",
        xaxis=dict(title=x_title, rangeslider=dict(visible=True)),
        yaxis=dict(title="Count"),
        template="plotly_white",
        margin=dict(l=70, r=70, t=60, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig1.update_layout(
        title=f"{title} | GT vs TP (+ TP/GT right axis)",
        xaxis=dict(title=x_title, rangeslider=dict(visible=True)),
        yaxis=dict(title="Count"),
        yaxis2=dict(title="TP/GT", overlaying="y", side="right"),
        template="plotly_white",
        margin=dict(l=70, r=70, t=60, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig2.update_layout(
        title=f"{title} | FP / FN / (FP+FN)",
        xaxis=dict(title=x_title, rangeslider=dict(visible=True)),
        yaxis=dict(title="Count"),
        template="plotly_white",
        margin=dict(l=70, r=70, t=60, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )

    out_html.parent.mkdir(parents=True, exist_ok=True)

    html0 = fig0.to_html(include_plotlyjs="cdn", full_html=False, div_id="plot_fig0")
    html1 = fig1.to_html(include_plotlyjs=False, full_html=False, div_id="plot_fig1")
    html2 = fig2.to_html(include_plotlyjs=False, full_html=False, div_id="plot_fig2")

    summary_html = make_day_summary_html(df_day)
    top_tables_html = make_top_tables_blocks_html(df_day, top_n=top_n, focus_col=focus_col)

    opt_lines = ["<option value='TOTAL'>TOTAL</option>"]
    for f in focus_values:
        opt_lines.append(
            f"<option value='{html.escape(_scope_for_focus_value(f))}'>focus={html.escape(f)}</option>"
        )
    options_html = "\n".join(opt_lines)

    focus_bar = f"""
<div style="border:1px solid #ddd; background:#f0f8ff; padding:10px; margin:10px 0 12px 0; font-family:Arial;">
  <div style="display:flex; gap:10px; align-items:center;">
    <div style="font-weight:700; color:#003366;">Focus filter</div>
    <select id="global_focus_select" style="padding:4px 10px; font-size:12px;">
      {options_html}
    </select>
    <div style="font-size:12px; color:#333;">(controls all charts + top table)</div>
  </div>
</div>
"""

    global_js = """
<script>
(function() {
  function showTable(scope) {
    var blocks = document.getElementsByClassName('top_table_block');
    for (var i=0;i<blocks.length;i++) blocks[i].style.display = 'none';
    for (var j=0;j<blocks.length;j++) {
      if ((blocks[j].getAttribute('data-scope') || '') === scope) {
        blocks[j].style.display = 'block';
        return;
      }
    }
  }

  function set_focus(scope) {
    var plotIds = ["plot_fig0","plot_fig1","plot_fig2"];
    plotIds.forEach(function(pid){
      var plot = document.getElementById(pid);
      if (!plot || !plot.data) return;

      var vis = [];
      for (var i=0;i<plot.data.length;i++) {
        var nm = (plot.data[i].name || "");
        vis.push(nm.startsWith(scope + " | "));
      }
      Plotly.restyle(plot, {visible: vis});
    });

    showTable(scope);
  }

  var sel = document.getElementById("global_focus_select");
  if (sel) {
    sel.addEventListener("change", function() {
      set_focus(this.value);
    });
  }

  // default
  set_focus("TOTAL");

  // click open FiftyOne for all plots
  function bindClick(plotId) {
    var plot = document.getElementById(plotId);
    if (!plot) return;
    plot.on('plotly_click', function(e) {
      var url = e?.points?.[0]?.customdata?.[0];
      if (url) window.open(url, '_blank');
    });
  }
  bindClick("plot_fig0");
  bindClick("plot_fig1");
  bindClick("plot_fig2");
})();
</script>
"""

    full = []
    full.append("<html><head><meta charset='utf-8'><title>Image-level</title></head><body>")
    full.append(focus_bar)
    full.append(summary_html)
    full.append(html0)
    full.append(html1)
    full.append(html2)
    full.append(top_tables_html)
    full.append(global_js)
    full.append("</body></html>")

    out_html.write_text("\n".join(full), encoding="utf-8")
    print(f"[SAVE][image-day] {out_html}")


# =========================
# Daily overview (unchanged)
# =========================
def write_daily_overview_html(
    df: pd.DataFrame,
    out_html: Path,
    day_to_file: Dict[str, str],
    title: str,
) -> None:
    daily = (
        df.groupby("capture_date", dropna=False)
        .agg(
            images=(("sample_id", "count") if "sample_id" in df.columns else ("filepath", "count")),
            hit=("hit_img", "sum"),
            miss=("miss_img", "sum"),
            false_alarm=("false_alarm_img", "sum"),
            correct_reject=("correct_reject_img", "sum"),
            gt_total=(("gt_count_img", "sum") if "gt_count_img" in df.columns else ("hit_img", "sum")),
            pred_total=(("pred_count_img", "sum") if "pred_count_img" in df.columns else ("hit_img", "sum")),
            tp_total=(("tp_img", "sum") if "tp_img" in df.columns else ("hit_img", "sum")),
            fp_total=(("fp_img", "sum") if "fp_img" in df.columns else ("hit_img", "sum")),
            fn_total=(("fn_img", "sum") if "fn_img" in df.columns else ("hit_img", "sum")),
        )
        .reset_index()
    ).sort_values("capture_date")

    daily["hit_rate"] = daily["hit"] / daily["images"].clip(lower=1)
    daily["error_rate"] = (daily["miss"] + daily["false_alarm"]) / daily["images"].clip(lower=1)

    denom = daily["tp_total"] + daily["fp_total"] + daily["fn_total"]
    daily["obj_error_rate"] = np.where(denom > 0, (daily["fp_total"] + daily["fn_total"]) / denom, np.nan)

    daily["avg_gt_per_img"] = (daily["gt_total"] / daily["images"].clip(lower=1)).astype("float")
    daily["avg_pred_per_img"] = (daily["pred_total"] / daily["images"].clip(lower=1)).astype("float")

    daily["drill_html"] = daily["capture_date"].map(day_to_file).fillna("")
    x = daily["capture_date"]

    fig = go.Figure()

    hover_pack = daily[
        [
            "drill_html",
            "images",
            "hit",
            "miss",
            "false_alarm",
            "correct_reject",
            "hit_rate",
            "error_rate",
            "gt_total",
            "pred_total",
            "tp_total",
            "fp_total",
            "fn_total",
            "avg_gt_per_img",
            "avg_pred_per_img",
            "obj_error_rate",
        ]
    ].values

    def daily_hovertemplate(series_name: str) -> str:
        return (
            "Date: %{x}<br>"
            "<b>" + series_name + "</b>: %{{y}}<br>"
            "Images: %{customdata[1]}<br>"
            "Hit/Miss/FA/CR: %{customdata[2]}/%{customdata[3]}/%{customdata[4]}/%{customdata[5]}<br>"
            "Hit rate (img): %{customdata[6]:.3f}<br>"
            "Error rate (img): %{customdata[7]:.3f}<br>"
            "GT total: %{customdata[8]} | Pred total: %{customdata[9]}<br>"
            "TP/FP/FN: %{customdata[10]}/%{customdata[11]}/%{customdata[12]}<br>"
            "avg GT/img: %{customdata[13]:.3f} | avg Pred/img: %{customdata[14]:.3f}<br>"
            "Obj error rate (FP+FN)/(TP+FP+FN): %{customdata[15]:.3f}<br>"
            "<b>Click</b> to drill down<br>"
            "<extra></extra>"
        )

    for col, name in [
        ("hit", "Hit (GT=1 & Pred=1)"),
        ("miss", "Miss (GT=1 & Pred=0)"),
        ("false_alarm", "False Alarm (GT=0 & Pred=1)"),
        ("correct_reject", "Correct Reject (GT=0 & Pred=0)"),
    ]:
        fig.add_trace(go.Bar(x=x, y=daily[col], name=name, customdata=hover_pack, hovertemplate=daily_hovertemplate(name)))

    fig.add_trace(go.Scatter(
        x=x, y=daily["hit_rate"],
        name="Hit rate (image)", mode="lines+markers", yaxis="y2",
        customdata=hover_pack, hovertemplate=daily_hovertemplate("Hit rate (image)"),
    ))
    fig.add_trace(go.Scatter(
        x=x, y=daily["error_rate"],
        name="Error rate (image) = (miss+FA)/images", mode="lines+markers", yaxis="y2",
        customdata=hover_pack, hovertemplate=daily_hovertemplate("Error rate (image)"),
    ))

    fig.add_trace(go.Scatter(
        x=x, y=daily["images"], mode="text",
        text=[f"{int(v)}" for v in daily["images"].fillna(0)],
        textposition="top center", showlegend=False, hoverinfo="skip",
    ))

    fig.update_layout(
        title=title,
        barmode="stack",
        xaxis=dict(title="capture_date", rangeslider=dict(visible=True)),
        yaxis=dict(title="Images (stacked by outcome)"),
        yaxis2=dict(title="Rate", overlaying="y", side="right", range=[0, 1]),
        template="plotly_white",
        margin=dict(l=70, r=70, t=60, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )

    post_script = """
    document.addEventListener("DOMContentLoaded", function() {
        var plot = document.getElementsByClassName('js-plotly-plot')[0];
        if (!plot) return;

        plot.on('plotly_click', function(e) {
            var rel = e?.points?.[0]?.customdata?.[0];
            if (rel) window.open(rel, '_blank');
        });
    });
    """

    out_html.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(out_html, include_plotlyjs="cdn", full_html=True, post_script=post_script)
    print(f"[SAVE][daily] {out_html}")

def write_minutely_swd_overview_html(
    df: pd.DataFrame,
    out_html: Path,
    title: str,
    focus_col: str = "focus",
    pred_col: str = "pred_count_img",
    gt_col: str = "gt_count_img",
) -> None:

    d = df.copy()
    d = ensure_time_cols(d)
    d = ensure_event_cols(d)

    if "capture_datetime" not in d.columns:
        return

    d["capture_datetime"] = pd.to_datetime(d["capture_datetime"], errors="coerce")
    d = d[d["capture_datetime"].notna()].copy()

    if focus_col not in d.columns:
        d[focus_col] = "unknown"
    d[focus_col] = d[focus_col].astype(str).fillna("unknown")

    d[pred_col] = pd.to_numeric(d.get(pred_col, 0), errors="coerce").fillna(0)
    d[gt_col] = pd.to_numeric(d.get(gt_col, 0), errors="coerce").fillna(0)

    # minute bucket
    d["minute"] = d["capture_datetime"].dt.floor("min")

    # drilldown → day page
    d["capture_date"] = d["capture_datetime"].dt.date.astype(str)
    d["drill_html"] = d["capture_date"].apply(lambda x: f"image_level_{x}.html")

    # ---------- aggregation ----------
    def agg(dd):
        return (
            dd.groupby("minute")
            .agg(
                pred=(pred_col, "sum"),
                gt=(gt_col, "sum"),
                images=("filepath", "count"),
                drill_html=("drill_html", "first"),
            )
            .reset_index()
            .sort_values("minute")
        )

    total_m = agg(d)
    focus_values = sorted(d[focus_col].unique().tolist())

    fig = go.Figure()

    # ---------- TOTAL ----------
    fig.add_trace(
        go.Scatter(
            x=total_m["minute"],
            y=total_m["pred"],
            mode="lines+markers",
            name="TOTAL | Pred/min",
            customdata=total_m[["images", "drill_html"]].values,
            hovertemplate=
                "Minute: %{x}<br>"
                "<b>Pred/min</b>: %{y}<br>"
                "Images: %{customdata[0]}<extra></extra>",
            visible=True,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=total_m["minute"],
            y=total_m["gt"],
            mode="lines",
            line=dict(dash="dash"),
            name="TOTAL | GT/min",
            customdata=total_m[["images", "drill_html"]].values,
            hovertemplate=
                "Minute: %{x}<br>"
                "<b>GT/min</b>: %{y}<br>"
                "Images: %{customdata[0]}<extra></extra>",
            visible=True,
        )
    )

    # ---------- per focus ----------
    for f in focus_values:
        sub = d[d[focus_col] == f]
        m = agg(sub)

        fig.add_trace(
            go.Scatter(
                x=m["minute"],
                y=m["pred"],
                mode="lines+markers",
                name=f"focus={f} | Pred/min",
                customdata=m[["images", "drill_html"]].values,
                visible=False,
            )
        )

        fig.add_trace(
            go.Scatter(
                x=m["minute"],
                y=m["gt"],
                mode="lines",
                line=dict(dash="dash"),
                name=f"focus={f} | GT/min",
                customdata=m[["images", "drill_html"]].values,
                visible=False,
            )
        )

    # ---------- layout ----------
    fig.update_layout(
        title=title,
        xaxis=dict(title="minute", rangeslider=dict(visible=True)),
        yaxis=dict(title="SWD count per minute"),
        template="plotly_white",
    )

    # ---------- dropdown ----------
    buttons = []

    n = len(fig.data)

    # TOTAL = trace 0,1
    vis_total = [False]*n
    vis_total[0] = True
    vis_total[1] = True

    buttons.append(dict(label="TOTAL", method="update", args=[{"visible": vis_total}]))

    # per focus = 每2条一组
    trace_idx = 2
    for f in focus_values:
        vis = [False]*n
        vis[trace_idx] = True
        vis[trace_idx+1] = True
        buttons.append(
            dict(label=f"focus={f}", method="update", args=[{"visible": vis}])
        )
        trace_idx += 2

    fig.update_layout(
        updatemenus=[dict(buttons=buttons, x=0.01, y=1.15, active=0)]
    )

    # ---------- click → day html ----------
    post_script = """
    document.addEventListener("DOMContentLoaded", function() {
        var plot = document.getElementsByClassName('js-plotly-plot')[0];
        if (!plot) return;

        plot.on('plotly_click', function(e) {
            var rel = e?.points?.[0]?.customdata?.[1];
            if (rel) window.open(rel, '_blank');
        });
    });
    """

    out_html.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(
        out_html,
        include_plotlyjs="cdn",
        full_html=True,
        post_script=post_script,
    )

    print("[SAVE][minutely]", out_html)


# =========================
# Per-CSV processing
# =========================
def process_one_csv(csv_path: Path) -> None:
    df = pd.read_csv(csv_path)

    if "filepath" not in df.columns:
        print(f"[SKIP] missing filepath: {csv_path}")
        return

    df = ensure_time_cols(df)
    df = ensure_event_cols(df)

    report_dir = OUT_ROOT / csv_path.parent.parent.name / csv_path.parent.name / safe_name(csv_path.stem)
    report_dir.mkdir(parents=True, exist_ok=True)

    day_to_file: Dict[str, str] = {}
    for day, g in df.groupby("capture_date"):
        day = str(day)
        out_day_html = report_dir / f"image_level_{day}.html"
        write_image_level_day_html(
            df_day=g,
            out_html=out_day_html,
            title=f"{csv_path.stem} | {day}",
            top_n=TOP_N,
            focus_col=FOCUS_COL,
        )
        day_to_file[day] = out_day_html.name

    out_daily_html = report_dir / "daily_overview.html"
    write_daily_overview_html(
        df=df,
        out_html=out_daily_html,
        day_to_file=day_to_file,
        title=f"{csv_path.stem} | Daily overview (click bars/lines to drill down)",
    )
    
    out_minutely_html = report_dir / "MINUTELY_overview.html"

    write_minutely_swd_overview_html(
        df=df,
        out_html=out_minutely_html,
        title=f"{csv_path.stem} | MINUTELY SWD count",
        focus_col=FOCUS_COL,
    )


    print(f"[DONE] report_dir = {report_dir}")


# =========================
# Main
# =========================
def main():
    csvs = find_per_image_csvs(PER_IMAGE_DIR)
    print("[INFO] Found CSVs:", len(csvs))
    if LIMIT is not None:
        csvs = csvs[:LIMIT]
        print("[INFO] Using first:", len(csvs))

    for p in csvs:
        print("\n" + "=" * 90)
        print("[CSV]", p)
        process_one_csv(p)

    print("\nAll done.")


if __name__ == "__main__":
    main()


[INFO] Found CSVs: 882

[CSV] _eval_exports_per_images/sahi_null_v3/sahi_null_v3_ms1_0605__0621_40_ok/yolo11l_20pct_null_images_add_rawData_batch_16_final/image_level_yolo11l_20pct_null_images_add_rawData_batch_16_final__c50__iou50.csv
[SAVE][image-day] _eval_exports_per_images/sahi_null_v3/_prod_kiss_reports/sahi_null_v3_ms1_0605__0621_40_ok/yolo11l_20pct_null_images_add_rawData_batch_16_final/image_level_yolo11l_20pct_null_images_add_rawData_batch_16_final__c50__iou50/image_level_2024-06-06.html
[SAVE][image-day] _eval_exports_per_images/sahi_null_v3/_prod_kiss_reports/sahi_null_v3_ms1_0605__0621_40_ok/yolo11l_20pct_null_images_add_rawData_batch_16_final/image_level_yolo11l_20pct_null_images_add_rawData_batch_16_final__c50__iou50/image_level_2024-06-07.html
[SAVE][image-day] _eval_exports_per_images/sahi_null_v3/_prod_kiss_reports/sahi_null_v3_ms1_0605__0621_40_ok/yolo11l_20pct_null_images_add_rawData_batch_16_final/image_level_yolo11l_20pct_null_images_add_rawData_batch_16_final__c5